In [2]:
import yaml
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy import text


pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

# Leer credenciales del arhivo YAML
with open("credenciales.yaml", "r") as file:
     config = yaml.safe_load(file)

db = config["mysql"]

# Crear conexión
# Su función es crear un Engine, que es el objeto que SQLAlchemy utiliza para comunicarse con la base de datos.
engine = create_engine(                                     
    f"mysql+pymysql://{db['user']}:{db['password']}@"
    f"{db['host']}:{db['port']}/{db['database']}"
)

##### ANALISIS DE TEXTO

##### 5-1 Introduccion

Se utilizaran datos de reseñas de un parque de diversiones. El objetivo es aprender herramientas para limpieza y análisis básico de texto. 


##### 5-2 Qué es el analisis de texto

Los datos de tipo texto pueden ser: 

$\cdot$ Estructurados: 		Datos de formularios por ejemplo

$\cdot$ No estructurados:	Datos extraídos directamente de libros, revistas, páginas web.

$\cdot$ Semi estructurados:	Tienen parte estructurada y no estructurada como los correos electronicos.


Hay 2 tipos de análisis de texto:

$\cdot$ Cualitativo:	Leer el texto y hacer un resumen del contenido.

$\cdot$ Cuantitativo:	Categorización, extracción de información. SQL nos servira para este análisis.

Metas:

$\cdot$ Extracción de texto: Extraer el numero de estrellas que dejo una reseña.

$\cdot$ Categorización: Categorizar las reseñas.

$\cdot$ Análisis de sentimiento: Si al reseña fue positiva, negativa o neutra. SQL no es la mejor opción para realizar este análisis.

##### 5-3 Por que sí usar SQL

Si se cuenta con una gran cantidad de datos y están disponibles directamente en una base relacional. Es una herramienta adecuada para tareas de conteo
o frecuencias, busqueda de palabras clave, limpiar y estructurar datos.

##### 5-4 SQL no es bueno para trabajar con texto

Si la base es muy pequeña, si la consultas requieren ser ser ejecutadas en el menor tiempo posible hay alternativas como Elastic Search, 
si el objetivo es leer todo el texto y entregar un resumen.

##### 5-5 Caracteristicas del Texto


In [13]:
# Mostrar las primeras 5 filas de la tabla disneyrew del esquema analisistexto

query = """
select * 
from analisistexto.disneyrew
limit 5;
"""
df = pd.read_sql_query(text(query), engine)
with pd.option_context(
    'display.max_colwidth', None,
    'display.max_columns', None
):
    display(df.head(10))

,disneyrew_id,Information
0,1,"Rating: 4, Year_Month: 2019-4, Reviewer_Location: Australia, Review_Text: If you've ever been to Disneyland anywhere you'll find Disneyland Hong Kong very similar in the layout when you walk into main street! It has a very familiar feel. One of the rides its a Small World is absolutely fabulous and worth doing. The day we visited was fairly hot and relatively busy but the queues moved fairly well. Branch: Disneyland_HongKong"
1,2,"Rating: 4, Year_Month: 2019-5, Reviewer_Location: Philippines, Review_Text: Its been a while since d last time we visit HK Disneyland .. Yet, this time we only stay in Tomorrowland .. AKA Marvel land!Now they have Iron Man Experience n d Newly open Ant Man n d Wasp!!Ironman .. Great feature n so Exciting, especially d whole scenery of HK (HK central area to Kowloon)!Antman .. Changed by previous Buzz lightyear! More or less d same, but I'm expecting to have something most!!However, my boys like it!!Space Mountain .. Turns into Star Wars!! This 1 is Great!!!For cast members (staffs) .. Felt bit MINUS point from before!!! Just dun feel like its a Disney brand!! Seems more local like Ocean Park or even worst!!They got no SMILING face, but just wanna u to enter n attraction n leave!!Hello this is supposed to be Happiest Place on Earth brand!! But, just really Dont feel it!!Bakery in Main Street now have more attractive delicacies n Disney theme sweets .. These are Good Points!!Last, they also have Starbucks now inside the theme park!! Branch: Disneyland_HongKong"
2,3,"Rating: 4, Year_Month: 2019-4, Reviewer_Location: United Arab Emirates, Review_Text: Thanks God it wasn t too hot or too humid when I was visiting the park otherwise it would be a big issue (there is not a lot of shade).I have arrived around 10:30am and left at 6pm. Unfortunately I didn t last until evening parade, but 8.5 hours was too much for me.There is plenty to do and everyone will find something interesting for themselves to enjoy.It wasn t extremely busy and the longest time I had to queue for certain attractions was 45 minutes (which is really not that bad).Although I had an amazing time, I felt a bit underwhelmed with choice of rides and attractions. The park itself is quite small (I was really expecting something grand even the main castle which was closed by the way was quite small).The food options are good, few coffee shops (including Starbucks) and plenty of gift shops. There was no issue with toilets as they are everywhere.All together it was a great day out and I really enjoyed it. Branch: Disneyland_HongKong"
3,4,"Rating: 4, Year_Month: 2019-4, Reviewer_Location: Australia, Review_Text: HK Disneyland is a great compact park. Unfortunately there is quite a bit of maintenance work going on at present so a number of areas are closed off (including the famous castle) If you go midweek, it is not too crowded and certainly no where near as bus as LA Disneyland. We did notice on this visit that prices for food, drinks etc have really gone through the roof so be prepared to pay top dollar for snacks (and avoid the souvenir shops if you can) Regardless, kids will love it. Branch: Disneyland_HongKong"
4,5,"Rating: 4, Year_Month: 2019-4, Reviewer_Location: United Kingdom, Review_Text: the location is not in the city, took around 1 hour from Kowlon, my kids like disneyland so much, everything is fine. but its really crowded and hot in Hong Kong Branch: Disneyland_HongKong"
